# SpMVBench - CUDA build + correctness on Colab

Runs the full CUDA path on the Colab GPU:

1. clones the public repo at the base commit,
2. applies the CUDA compile fixes + `gamma_n`-tolerance CUDA tests,
3. prints GPU/driver/nvcc/compute-capability,
4. configures with `SPMVBENCH_ENABLE_CUDA=ON` and the matching
   `CMAKE_CUDA_ARCHITECTURES`, builds, and runs `ctest`.

Choose **Runtime > Change runtime type > Hardware accelerator: GPU**
first, then **Runtime > Run all**.


In [ ]:
import base64, os, pathlib, subprocess

REPO = "/content/SpMVBench"
BASE = "e32e0c39b45d71f74527142978c1b6533625f167"
PATCH_B64 = """
ZGlmZiAtLWdpdCBhL3NyYy9jdWRhL2N1c3BhcnNlX2Jhc2VsaW5lLmN1IGIvc3JjL2N1ZGEvY3VzcGFyc2VfYmFzZWxpbmUuY3UKaW5kZXggM2U3ODYyNC4uNjhmOTdhYiAxMDA2NDQKLS0tIGEvc3JjL2N1ZGEvY3VzcGFyc2VfYmFzZWxpbmUuY3UKKysrIGIvc3JjL2N1ZGEvY3VzcGFyc2VfYmFzZWxpbmUuY3UKQEAgLTYsNiArNiw3IEBACiAjaW5jbHVkZSAic3BtdmJlbmNoL3NwbXZfY3VkYS5jdWgiCiAKICNpbmNsdWRlIDxhbGdvcml0aG0+CisjaW5jbHVkZSA8Y3N0ZGRlZj4KICNpbmNsdWRlIDx2ZWN0b3I+CiAKIG5hbWVzcGFjZSBzcG12YmVuY2ggewpkaWZmIC0tZ2l0IGEvc3JjL2N1ZGEvY3VzcGFyc2Vfc3BtbS5jdSBiL3NyYy9jdWRhL2N1c3BhcnNlX3NwbW0uY3UKaW5kZXggMThiM2VhZS4uMDJhMDdlNSAxMDA2NDQKLS0tIGEvc3JjL2N1ZGEvY3VzcGFyc2Vfc3BtbS5jdQorKysgYi9zcmMvY3VkYS9jdXNwYXJzZV9zcG1tLmN1CkBAIC00LDYgKzQsNyBAQAogI2luY2x1ZGUgInNwbXZiZW5jaC9zcG1tX2N1ZGEuY3VoIgogCiAjaW5jbHVkZSA8YWxnb3JpdGhtPgorI2luY2x1ZGUgPGNzdGRkZWY+CiAjaW5jbHVkZSA8dmVjdG9yPgogCiBuYW1lc3BhY2Ugc3BtdmJlbmNoIHsKZGlmZiAtLWdpdCBhL3NyYy9jdWRhL3NwbW1fY3VkYS5jdSBiL3NyYy9jdWRhL3NwbW1fY3VkYS5jdQppbmRleCBhNjIxMjZiLi5jODAyOWVkIDEwMDY0NAotLS0gYS9zcmMvY3VkYS9zcG1tX2N1ZGEuY3UKKysrIGIvc3JjL2N1ZGEvc3BtbV9jdWRhLmN1CkBAIC0xMCw2ICsxMCw3IEBACiAKICNpbmNsdWRlIDxhbGdvcml0aG0+CiAjaW5jbHVkZSA8Y21hdGg+CisjaW5jbHVkZSA8Y3N0ZGRlZj4KICNpbmNsdWRlIDxzdGRleGNlcHQ+CiAjaW5jbHVkZSA8c3RyaW5nPgogI2luY2x1ZGUgPHZlY3Rvcj4KZGlmZiAtLWdpdCBhL3NyYy9jdWRhL3NwbXZfY3VkYS5jdSBiL3NyYy9jdWRhL3NwbXZfY3VkYS5jdQppbmRleCA3MWM4YThhLi40OGUzOWE3IDEwMDY0NAotLS0gYS9zcmMvY3VkYS9zcG12X2N1ZGEuY3UKKysrIGIvc3JjL2N1ZGEvc3Btdl9jdWRhLmN1CkBAIC04LDYgKzgsNyBAQAogCiAjaW5jbHVkZSA8YWxnb3JpdGhtPgogI2luY2x1ZGUgPGNtYXRoPgorI2luY2x1ZGUgPGNzdGRkZWY+CiAjaW5jbHVkZSA8c3RkZXhjZXB0PgogCiAjaW5jbHVkZSAic3BtdmJlbmNoL2Jjc3IuaHBwIgpAQCAtMTksNiArMjAsMTUgQEAgbmFtZXNwYWNlIHsKIAogY29uc3RleHByIGludCBrV2FycCA9IDMyOwogCisvLyBCbG9jay1zaGFwZSB0YWdzIGZvciB0aGUgdHdvIEJDU1IgaW5zdGFudGlhdGlvbnMuIFRoZXkgbXVzdCBsaXZlIGF0CisvLyBuYW1lc3BhY2Ugc2NvcGU6IGEgbG9jYWwgY2xhc3MgaXMgbm90IGFsbG93ZWQgdG8gaGF2ZSBzdGF0aWMgZGF0YSBtZW1iZXJzLgorc3RydWN0IFRhZzIyIHsKKyAgc3RhdGljIGNvbnN0ZXhwciBpbnQgUiA9IDIsIEMgPSAyOworfTsKK3N0cnVjdCBUYWc0NCB7CisgIHN0YXRpYyBjb25zdGV4cHIgaW50IFIgPSA0LCBDID0gNDsKK307CisKIHRlbXBsYXRlIDx0eXBlbmFtZSBBY2M+CiBfX2RldmljZV9fIF9fZm9yY2VpbmxpbmVfXyBBY2Mgd2FycF9yZWR1Y2Vfc3VtKEFjYyB2LCB1bnNpZ25lZCBtYXNrKSB7CiAjcHJhZ21hIHVucm9sbApAQCAtMjIyLDEyICsyMzIsNiBAQCB2b2lkIFNwbXZSdW5uZXI8VCwgQWNjPjo6dXBsb2FkX21hdHJpeCgpIHsKICAgICBjb3B5X3RvX2RldmljZSgqdiwgYi52YWx1ZXMpOwogICAgICpvdXQgPSBEZXZpY2VCY3NyPFQ+eypyLCAqYywgKnYsIG0sIG4sIFIsIEMsIGIubWIsIGIuYmxvY2tfbm56fTsKICAgfTsKLSAgc3RydWN0IFRhZzIyIHsKLSAgICBzdGF0aWMgY29uc3RleHByIGludCBSID0gMiwgQyA9IDI7Ci0gIH07Ci0gIHN0cnVjdCBUYWc0NCB7Ci0gICAgc3RhdGljIGNvbnN0ZXhwciBpbnQgUiA9IDQsIEMgPSA0OwotICB9OwogICB1cGxvYWRfYmNzcihUYWcyMnt9LCAmZF9iMl92YWx1ZXNfLCAmZF9iMl9jb2xpbmRfLCAmZF9iMl9yb3dwdHJfLCAmZF9iY3NyMl8pOwogICB1cGxvYWRfYmNzcihUYWc0NHt9LCAmZF9iNF92YWx1ZXNfLCAmZF9iNF9jb2xpbmRfLCAmZF9iNF9yb3dwdHJfLCAmZF9iY3NyNF8pOwogCmRpZmYgLS1naXQgYS90ZXN0cy9DTWFrZUxpc3RzLnR4dCBiL3Rlc3RzL0NNYWtlTGlzdHMudHh0CmluZGV4IGVjYTNjMWEuLmYxZTJmYzcgMTAwNjQ0Ci0tLSBhL3Rlc3RzL0NNYWtlTGlzdHMudHh0CisrKyBiL3Rlc3RzL0NNYWtlTGlzdHMudHh0CkBAIC0xNywxMSArMTcsMjEgQEAgYWRkX2V4ZWN1dGFibGUoc3BtdmJlbmNoX3Rlc3RzCiAgIHRlc3Rfc3Btdl9jb3JyZWN0bmVzcy5jcHAKICAgdGVzdF9zcG1tX2NvcnJlY3RuZXNzLmNwcAogICB0ZXN0X3NlbGVjdC5jcHAKKyAgdGVzdF9jdWRhX2NvcnJlY3RuZXNzLmNwcAogKQogCiB0YXJnZXRfbGlua19saWJyYXJpZXMoc3BtdmJlbmNoX3Rlc3RzIFBSSVZBVEUgc3BtdmJlbmNoX2NvcmUgR1Rlc3Q6Omd0ZXN0X21haW4pCiB0YXJnZXRfY29tcGlsZV9kZWZpbml0aW9ucyhzcG12YmVuY2hfdGVzdHMgUFJJVkFURQogICBTUE1WQkVOQ0hfU09VUkNFX0RJUj0iJHtDTUFLRV9TT1VSQ0VfRElSfSIpCiAKK2lmKFNQTVZCRU5DSF9DVURBX09OKQorICAjIFRoZSBob3N0LWNvbXBpbGVkIHRlc3QgZmlsZSBpbmNsdWRlcyB0aGUgQ1VEQSBoZWFkZXJzIGFuZCBjYWxscyBpbnRvIHRoZQorICAjIGV4cGxpY2l0bHkgaW5zdGFudGlhdGVkIFNwbXZSdW5uZXIvU3BNTSBzeW1ib2xzIGZyb20gc3BtdmJlbmNoX2NvcmUuCisgIHRhcmdldF9saW5rX2xpYnJhcmllcyhzcG12YmVuY2hfdGVzdHMgUFJJVkFURSBDVURBOjpjdWRhcnQgQ1VEQTo6Y3VzcGFyc2UpCisgIHRhcmdldF9jb21waWxlX2RlZmluaXRpb25zKHNwbXZiZW5jaF90ZXN0cyBQUklWQVRFIFNQTVZCRU5DSF9FTkFCTEVfQ1VEQT0xKQorICBzZXRfdGFyZ2V0X3Byb3BlcnRpZXMoc3BtdmJlbmNoX3Rlc3RzIFBST1BFUlRJRVMKKyAgICBDVURBX1JFU09MVkVfREVWSUNFX1NZTUJPTFMgT04pCitlbmRpZigpCisKIGluY2x1ZGUoR29vZ2xlVGVzdCkKIGd0ZXN0X2Rpc2NvdmVyX3Rlc3RzKHNwbXZiZW5jaF90ZXN0cyBESVNDT1ZFUllfVElNRU9VVCA2MCkKZGlmZiAtLWdpdCBhL3Rlc3RzL3Rlc3RfY3VkYV9jb3JyZWN0bmVzcy5jcHAgYi90ZXN0cy90ZXN0X2N1ZGFfY29ycmVjdG5lc3MuY3BwCm5ldyBmaWxlIG1vZGUgMTAwNjQ0CmluZGV4IDAwMDAwMDAuLjU5YWZjYzAKLS0tIC9kZXYvbnVsbAorKysgYi90ZXN0cy90ZXN0X2N1ZGFfY29ycmVjdG5lc3MuY3BwCkBAIC0wLDAgKzEsMjYwIEBACisvLyBDVURBIGNvcnJlY3RuZXNzOiBldmVyeSBHUFUgU3BNViBrZXJuZWwgYW5kIHRoZSB0aWxlZC9idWNrZXRlZCBTcE1NIGtlcm5lbHMKKy8vIGFyZSBjb21wYXJlZCBhZ2FpbnN0IHRoZSBDUFUgcmVmZXJlbmNlIG9uIGV2ZXJ5IHN5bnRoZXRpYyBtYXRyaXgsIGluIGZwMzIKKy8vIGFuZCBmcDY0LgorLy8KKy8vIFRvbGVyYW5jZXMgYXJlIGRlcml2ZWQgZnJvbSB0aGUgY2xhc3NpY2FsIGdhbW1hX24gYm91bmQgcmF0aGVyIHRoYW4gdHVuZWQ6CisvLyBmb3IgdHdvIGRpZmZlcmVudCBzdW1tYXRpb24gb3JkZXJzIG9mIHRoZSBzYW1lIG4gdGVybXMsCisvLyAgICAgfHMxIC0gczJ8IDw9IHxzMSAtIGV4YWN0fCArIHxzMiAtIGV4YWN0fCA8PSAyICogZ2FtbWFfbiAqIHN1bV9qIHxhX2lqCisvLyAgICAgeF9qfAorLy8gd2l0aCBnYW1tYV9uID0gbip1IC8gKDEgLSBuKnUpIGFuZCB1IHRoZSB1bml0IHJvdW5kb2ZmIG9mIHRoZSBhY2N1bXVsYXRvci4KKy8vIFRoZSAyeCBhY2NvdW50cyBmb3IgY29tcGFyaW5nIHR3byBpbmRlcGVuZGVudCBzdW1tYXRpb24gb3JkZXJzOyBubyBvdGhlcgorLy8gc2xhY2sgaXMgYWRkZWQuCisKKyNpbmNsdWRlIDxndGVzdC9ndGVzdC5oPgorCisjaWZkZWYgU1BNVkJFTkNIX0VOQUJMRV9DVURBCisKKyNpbmNsdWRlIDxhbGdvcml0aG0+CisjaW5jbHVkZSA8Y21hdGg+CisjaW5jbHVkZSA8Y3N0ZGRlZj4KKyNpbmNsdWRlIDxjc3RkaW8+CisjaW5jbHVkZSA8ZmlsZXN5c3RlbT4KKyNpbmNsdWRlIDxyYW5kb20+CisjaW5jbHVkZSA8c3RyaW5nPgorI2luY2x1ZGUgPHR5cGVfdHJhaXRzPgorI2luY2x1ZGUgPHZlY3Rvcj4KKworI2luY2x1ZGUgInNwbXZiZW5jaC9jc3IuaHBwIgorI2luY2x1ZGUgInNwbXZiZW5jaC9tdHhfaW8uaHBwIgorI2luY2x1ZGUgInNwbXZiZW5jaC9zcG1tX2N1ZGEuY3VoIgorI2luY2x1ZGUgInNwbXZiZW5jaC9zcG12X2NwdS5ocHAiCisjaW5jbHVkZSAic3BtdmJlbmNoL3NwbXZfY3VkYS5jdWgiCisKK25hbWVzcGFjZSB7CisKK3N0ZDo6c3RyaW5nIGRhdGFfZGlyKCkgeworICByZXR1cm4gc3RkOjpzdHJpbmcoU1BNVkJFTkNIX1NPVVJDRV9ESVIpICsgIi9kYXRhIjsKK30KKworLy8gRXhhY3RseSB0aGUgbWF0cmljZXMgcHJvZHVjZWQgYnkgdG9vbHMvZ2VuX21hdHJpY2VzLnB5LgorY29uc3Qgc3RkOjp2ZWN0b3I8c3RkOjpzdHJpbmc+JiBzeW50aGV0aWNfbWF0cmljZXMoKSB7CisgIHN0YXRpYyBjb25zdCBzdGQ6OnZlY3RvcjxzdGQ6OnN0cmluZz4gbmFtZXMgPSB7CisgICAgICAiYmFuZGVkXzEwMGsiLCAidW5pZm9ybV9yYW5kb20iLCAicG93ZXJsYXdfc2tldyIsICJkaWFnX2RvbWluYW50X3NtYWxsIiwKKyAgICAgICJ0aW55XzN4MyJ9OworICByZXR1cm4gbmFtZXM7Cit9CisKK3RlbXBsYXRlIDx0eXBlbmFtZSBUPgorZG91YmxlIHVuaXRfcm91bmRvZmYoKSB7CisgIHJldHVybiBzdGQ6OmlzX3NhbWU8VCwgZmxvYXQ+Ojp2YWx1ZSA/IHN0ZDo6bGRleHAoMS4wLCAtMjQpCisgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICA6IHN0ZDo6bGRleHAoMS4wLCAtNTMpOworfQorCitkb3VibGUgZ2FtbWFfbihpbnQgbiwgZG91YmxlIHUpIHsKKyAgY29uc3QgZG91YmxlIG51ID0gc3RhdGljX2Nhc3Q8ZG91YmxlPihuKSAqIHU7CisgIHJldHVybiBudSAvICgxLjAgLSBudSk7Cit9CisKK3RlbXBsYXRlIDx0eXBlbmFtZSBUPgorc3RkOjp2ZWN0b3I8VD4gbWFrZV94KGludCBuLCB1bnNpZ25lZCBzZWVkKSB7CisgIHN0ZDo6bXQxOTkzNyBybmcoc2VlZCk7CisgIHN0ZDo6dW5pZm9ybV9yZWFsX2Rpc3RyaWJ1dGlvbjxkb3VibGU+IGJpdCgwLjAsIDEuMCk7CisgIHN0ZDo6dW5pZm9ybV9yZWFsX2Rpc3RyaWJ1dGlvbjxkb3VibGU+IGV4cG9uZW50KC02LjAsIDYuMCk7CisgIHN0ZDo6dmVjdG9yPFQ+IHgoc3RhdGljX2Nhc3Q8c3RkOjpzaXplX3Q+KG4pKTsKKyAgZm9yIChpbnQgaSA9IDA7IGkgPCBuOyArK2kpIHsKKyAgICBjb25zdCBkb3VibGUgc2lnbiA9IGJpdChybmcpIDwgMC41ID8gLTEuMCA6IDEuMDsKKyAgICB4W3N0YXRpY19jYXN0PHN0ZDo6c2l6ZV90PihpKV0gPQorICAgICAgICBzdGF0aWNfY2FzdDxUPihzaWduICogc3RkOjpwb3coMTAuMCwgZXhwb25lbnQocm5nKSkpOworICB9CisgIHJldHVybiB4OworfQorCit0ZW1wbGF0ZSA8dHlwZW5hbWUgVCwgdHlwZW5hbWUgQWNjPgordm9pZCBjaGVja19zcG12KGNvbnN0IHN0ZDo6c3RyaW5nJiBtYXRyaXgsIGNvbnN0IHN0ZDo6c3RyaW5nJiBrZXJuZWwsCisgICAgICAgICAgICAgICAgY29uc3Qgc3BtdmJlbmNoOjpDc3JNYXRyaXg8VD4mIEEsIGNvbnN0IHN0ZDo6dmVjdG9yPFQ+JiB4LAorICAgICAgICAgICAgICAgIGNvbnN0IHN0ZDo6dmVjdG9yPEFjYz4mIHlfcmVmLCBjb25zdCBzdGQ6OnZlY3RvcjxBY2M+JiB5X2dwdSkgeworICBBU1NFUlRfRVEoeV9yZWYuc2l6ZSgpLCB5X2dwdS5zaXplKCkpOworICBBU1NFUlRfRVEoeV9yZWYuc2l6ZSgpLCBzdGF0aWNfY2FzdDxzdGQ6OnNpemVfdD4oQS5tKSk7CisgIGNvbnN0IGRvdWJsZSB1ID0gdW5pdF9yb3VuZG9mZjxBY2M+KCk7CisgIGRvdWJsZSB3b3JzdF9yYXRpbyA9IDAuMCwgd29yc3RfZGlmZiA9IDAuMCwgd29yc3RfYm91bmQgPSAwLjA7CisgIGludCB3b3JzdF9yb3cgPSAtMTsKKyAgZm9yIChpbnQgaSA9IDA7IGkgPCBBLm07ICsraSkgeworICAgIGRvdWJsZSBhYnNfc3VtID0gMC4wOworICAgIGZvciAoc3BtdmJlbmNoOjpJbmRleCBwID0gQS5yb3dwdHJbaV07IHAgPCBBLnJvd3B0cltpICsgMV07ICsrcCkgeworICAgICAgYWJzX3N1bSArPSBzdGQ6OmZhYnMoc3RhdGljX2Nhc3Q8ZG91YmxlPihBLnZhbHVlc1twXSkgKgorICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RhdGljX2Nhc3Q8ZG91YmxlPih4W0EuY29saW5kW3BdXSkpOworICAgIH0KKyAgICBjb25zdCBpbnQgbiA9IHN0YXRpY19jYXN0PGludD4oQS5yb3dwdHJbaSArIDFdIC0gQS5yb3dwdHJbaV0pOworICAgIGNvbnN0IGRvdWJsZSBib3VuZCA9IDIuMCAqIGdhbW1hX24obiwgdSkgKiBhYnNfc3VtICsgMWUtMzA7CisgICAgY29uc3QgZG91YmxlIGRpZmYgPSBzdGQ6OmZhYnMoc3RhdGljX2Nhc3Q8ZG91YmxlPih5X2dwdVtpXSkgLQorICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXRpY19jYXN0PGRvdWJsZT4oeV9yZWZbaV0pKTsKKyAgICBjb25zdCBkb3VibGUgcmF0aW8gPSBkaWZmIC8gYm91bmQ7CisgICAgaWYgKHJhdGlvID4gd29yc3RfcmF0aW8pIHsKKyAgICAgIHdvcnN0X3JhdGlvID0gcmF0aW87CisgICAgICB3b3JzdF9yb3cgPSBpOworICAgICAgd29yc3RfZGlmZiA9IGRpZmY7CisgICAgICB3b3JzdF9ib3VuZCA9IGJvdW5kOworICAgIH0KKyAgfQorICBFWFBFQ1RfTEUod29yc3RfcmF0aW8sIDEuMCkKKyAgICAgIDw8ICJtYXRyaXg9IiA8PCBtYXRyaXggPDwgIiBrZXJuZWw9IiA8PCBrZXJuZWwKKyAgICAgIDw8ICIgYWNjdW11bGF0b3I9IiA8PCBzcG12YmVuY2g6OmR0eXBlX25hbWU8QWNjPigpIDw8ICIgcm93PSIgPDwgd29yc3Rfcm93CisgICAgICA8PCAiIHxncHUtY3B1fD0iIDw8IHdvcnN0X2RpZmYgPDwgIiBib3VuZD0iIDw8IHdvcnN0X2JvdW5kCisgICAgICA8PCAiIHJhdGlvPSIgPDwgd29yc3RfcmF0aW87Cit9CisKK3RlbXBsYXRlIDx0eXBlbmFtZSBULCB0eXBlbmFtZSBBY2M+Cit2b2lkIHJ1bl9zcG12X21hdHJpeChjb25zdCBzdGQ6OnN0cmluZyYgbmFtZSkgeworICBjb25zdCBzdGQ6OnN0cmluZyBwYXRoID0gZGF0YV9kaXIoKSArICIvIiArIG5hbWUgKyAiLm10eCI7CisgIGlmICghc3RkOjpmaWxlc3lzdGVtOjpleGlzdHMocGF0aCkpIHsKKyAgICBHVEVTVF9TS0lQKCkgPDwgcGF0aCA8PCAiIG1pc3Npbmc7IHJ1biB0b29scy9nZW5fbWF0cmljZXMucHkiOworICB9CisgIGNvbnN0IGF1dG8gQSA9IHNwbXZiZW5jaDo6cmVhZF9tYXRyaXhfbWFya2V0PFQ+KHBhdGgpOworICBjb25zdCBzdGQ6OnZlY3RvcjxUPiB4ID0gbWFrZV94PFQ+KEEubiwgMTIzNHUpOworCisgIHN0ZDo6dmVjdG9yPEFjYz4geV9yZWYoc3RhdGljX2Nhc3Q8c3RkOjpzaXplX3Q+KEEubSkpOworICBzcG12YmVuY2g6OnNwbXZfY3NyX2NwdTxULCBBY2M+KEEsIHguZGF0YSgpLCB5X3JlZi5kYXRhKCkpOworCisgIHNwbXZiZW5jaDo6U3BtdlJ1bm5lcjxULCBBY2M+IHJ1bm5lcihBKTsKKyAgcnVubmVyLnNldF94KHgpOworCisgIGF1dG8gcnVuX2FuZF9jaGVjayA9IFsmXShzcG12YmVuY2g6Oktlcm5lbElkIGlkLCBpbnQgbGFuZXMsIGludCBibG9jaywKKyAgICAgICAgICAgICAgICAgICAgICAgICAgIGNvbnN0IHN0ZDo6c3RyaW5nJiBsYWJlbCkgeworICAgIHJ1bm5lci5ydW4oaWQsIC8qaXRlcnM9Ki8xLCBsYW5lcywgYmxvY2ssIC8qd2FybXVwPSovMCk7CisgICAgY29uc3Qgc3RkOjp2ZWN0b3I8QWNjPiBnb3QgPSBydW5uZXIucmVzdWx0KCk7CisgICAgY2hlY2tfc3BtdjxULCBBY2M+KG5hbWUsIGxhYmVsLCBBLCB4LCB5X3JlZiwgZ290KTsKKyAgfTsKKworICBydW5fYW5kX2NoZWNrKHNwbXZiZW5jaDo6S2VybmVsSWQ6OkNzclNjYWxhciwgMSwgMSwgImNzcl9zY2FsYXIiKTsKKyAgZm9yIChpbnQgbGFuZXMgOiB7MiwgNCwgOCwgMTYsIDMyfSkgeworICAgIHJ1bl9hbmRfY2hlY2soc3BtdmJlbmNoOjpLZXJuZWxJZDo6Q3NyVmVjdG9yLCBsYW5lcywgMSwKKyAgICAgICAgICAgICAgICAgICJjc3JfdmVjdG9yX2xhbmVzIiArIHN0ZDo6dG9fc3RyaW5nKGxhbmVzKSk7CisgIH0KKyAgcnVuX2FuZF9jaGVjayhzcG12YmVuY2g6Oktlcm5lbElkOjpDc3JXYXJwLCAxLCAxLCAiY3NyX3dhcnAiKTsKKyAgcnVuX2FuZF9jaGVjayhzcG12YmVuY2g6Oktlcm5lbElkOjpDc3JNZXJnZSwgMSwgMSwgImNzcl9tZXJnZSIpOworICBydW5fYW5kX2NoZWNrKHNwbXZiZW5jaDo6S2VybmVsSWQ6OkVsbCwgMSwgMSwgImVsbCIpOworICBydW5fYW5kX2NoZWNrKHNwbXZiZW5jaDo6S2VybmVsSWQ6OkJjc3IsIDEsIDIsICJiY3NyXzJ4MiIpOworICBydW5fYW5kX2NoZWNrKHNwbXZiZW5jaDo6S2VybmVsSWQ6OkJjc3IsIDEsIDQsICJiY3NyXzR4NCIpOworfQorCit0ZW1wbGF0ZSA8dHlwZW5hbWUgVCwgdHlwZW5hbWUgQWNjPgordm9pZCBjaGVja19zcG1tKGNvbnN0IHN0ZDo6c3RyaW5nJiBtYXRyaXgsIGNvbnN0IHN0ZDo6c3RyaW5nJiBrZXJuZWwsCisgICAgICAgICAgICAgICAgY29uc3Qgc3BtdmJlbmNoOjpDc3JNYXRyaXg8VD4mIEEsIGNvbnN0IHN0ZDo6dmVjdG9yPFQ+JiBCLAorICAgICAgICAgICAgICAgIGludCBrLCBjb25zdCBzdGQ6OnZlY3RvcjxBY2M+JiBZX3JlZiwKKyAgICAgICAgICAgICAgICBjb25zdCBzdGQ6OnZlY3RvcjxBY2M+JiBZX2dwdSkgeworICBBU1NFUlRfRVEoWV9yZWYuc2l6ZSgpLCBZX2dwdS5zaXplKCkpOworICBBU1NFUlRfRVEoWV9yZWYuc2l6ZSgpLCBzdGF0aWNfY2FzdDxzdGQ6OnNpemVfdD4oQS5tKSAqIGspOworICBjb25zdCBkb3VibGUgdSA9IHVuaXRfcm91bmRvZmY8QWNjPigpOworICBkb3VibGUgd29yc3RfcmF0aW8gPSAwLjAsIHdvcnN0X2RpZmYgPSAwLjAsIHdvcnN0X2JvdW5kID0gMC4wOworICBpbnQgd29yc3RfaSA9IC0xLCB3b3JzdF9qID0gLTE7CisgIGZvciAoaW50IGkgPSAwOyBpIDwgQS5tOyArK2kpIHsKKyAgICBjb25zdCBpbnQgbiA9IHN0YXRpY19jYXN0PGludD4oQS5yb3dwdHJbaSArIDFdIC0gQS5yb3dwdHJbaV0pOworICAgIGNvbnN0IGRvdWJsZSBnID0gZ2FtbWFfbihuLCB1KTsKKyAgICBmb3IgKGludCBqID0gMDsgaiA8IGs7ICsraikgeworICAgICAgZG91YmxlIGFic19zdW0gPSAwLjA7CisgICAgICBmb3IgKHNwbXZiZW5jaDo6SW5kZXggcCA9IEEucm93cHRyW2ldOyBwIDwgQS5yb3dwdHJbaSArIDFdOyArK3ApIHsKKyAgICAgICAgY29uc3QgaW50IGMgPSBBLmNvbGluZFtwXTsKKyAgICAgICAgYWJzX3N1bSArPSBzdGQ6OmZhYnMoCisgICAgICAgICAgICBzdGF0aWNfY2FzdDxkb3VibGU+KEEudmFsdWVzW3BdKSAqCisgICAgICAgICAgICBzdGF0aWNfY2FzdDxkb3VibGU+KEJbc3RhdGljX2Nhc3Q8c3RkOjpzaXplX3Q+KGMpICogayArIGpdKSk7CisgICAgICB9CisgICAgICBjb25zdCBkb3VibGUgYm91bmQgPSAyLjAgKiBnICogYWJzX3N1bSArIDFlLTMwOworICAgICAgY29uc3Qgc3RkOjpzaXplX3QgaWR4ID0gc3RhdGljX2Nhc3Q8c3RkOjpzaXplX3Q+KGkpICogayArIGo7CisgICAgICBjb25zdCBkb3VibGUgZGlmZiA9IHN0ZDo6ZmFicyhzdGF0aWNfY2FzdDxkb3VibGU+KFlfZ3B1W2lkeF0pIC0KKyAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXRpY19jYXN0PGRvdWJsZT4oWV9yZWZbaWR4XSkpOworICAgICAgY29uc3QgZG91YmxlIHJhdGlvID0gZGlmZiAvIGJvdW5kOworICAgICAgaWYgKHJhdGlvID4gd29yc3RfcmF0aW8pIHsKKyAgICAgICAgd29yc3RfcmF0aW8gPSByYXRpbzsKKyAgICAgICAgd29yc3RfaSA9IGk7CisgICAgICAgIHdvcnN0X2ogPSBqOworICAgICAgICB3b3JzdF9kaWZmID0gZGlmZjsKKyAgICAgICAgd29yc3RfYm91bmQgPSBib3VuZDsKKyAgICAgIH0KKyAgICB9CisgIH0KKyAgRVhQRUNUX0xFKHdvcnN0X3JhdGlvLCAxLjApCisgICAgICA8PCAibWF0cml4PSIgPDwgbWF0cml4IDw8ICIga2VybmVsPSIgPDwga2VybmVsCisgICAgICA8PCAiIGFjY3VtdWxhdG9yPSIgPDwgc3BtdmJlbmNoOjpkdHlwZV9uYW1lPEFjYz4oKSA8PCAiIGF0ICgiIDw8IHdvcnN0X2kKKyAgICAgIDw8ICIsIiA8PCB3b3JzdF9qIDw8ICIpIHxncHUtY3B1fD0iIDw8IHdvcnN0X2RpZmYKKyAgICAgIDw8ICIgYm91bmQ9IiA8PCB3b3JzdF9ib3VuZCA8PCAiIHJhdGlvPSIgPDwgd29yc3RfcmF0aW87Cit9CisKK3RlbXBsYXRlIDx0eXBlbmFtZSBULCB0eXBlbmFtZSBBY2M+Cit2b2lkIHJ1bl9zcG1tX21hdHJpeChjb25zdCBzdGQ6OnN0cmluZyYgbmFtZSwgaW50IGsgPSAzMikgeworICBjb25zdCBzdGQ6OnN0cmluZyBwYXRoID0gZGF0YV9kaXIoKSArICIvIiArIG5hbWUgKyAiLm10eCI7CisgIGlmICghc3RkOjpmaWxlc3lzdGVtOjpleGlzdHMocGF0aCkpIHsKKyAgICBHVEVTVF9TS0lQKCkgPDwgcGF0aCA8PCAiIG1pc3Npbmc7IHJ1biB0b29scy9nZW5fbWF0cmljZXMucHkiOworICB9CisgIGNvbnN0IGF1dG8gQSA9IHNwbXZiZW5jaDo6cmVhZF9tYXRyaXhfbWFya2V0PFQ+KHBhdGgpOworCisgIHN0ZDo6bXQxOTkzNyBybmcoNDMyMSk7CisgIHN0ZDo6dW5pZm9ybV9yZWFsX2Rpc3RyaWJ1dGlvbjxkb3VibGU+IGRpc3QoLTEuMCwgMS4wKTsKKyAgc3RkOjp2ZWN0b3I8VD4gQihzdGF0aWNfY2FzdDxzdGQ6OnNpemVfdD4oQS5uKSAqIGspOworICBmb3IgKGF1dG8mIHYgOiBCKQorICAgIHYgPSBzdGF0aWNfY2FzdDxUPihkaXN0KHJuZykpOworCisgIHN0ZDo6dmVjdG9yPEFjYz4gWV9yZWYoc3RhdGljX2Nhc3Q8c3RkOjpzaXplX3Q+KEEubSkgKiBrKTsKKyAgc3BtdmJlbmNoOjpzcG1tX2Nzcl9jcHU8VCwgQWNjPihBLCBCLmRhdGEoKSwgWV9yZWYuZGF0YSgpLCBrKTsKKworICBmb3IgKGludCB0aWxlIDogezE2LCAzMiwgNjR9KSB7CisgICAgZm9yIChib29sIGJhbGFuY2VkIDoge2ZhbHNlLCB0cnVlfSkgeworICAgICAgc3RkOjp2ZWN0b3I8VD4gWTsKKyAgICAgIGlmIGNvbnN0ZXhwciAoc3RkOjppc19zYW1lPFQsIGZsb2F0Pjo6dmFsdWUpIHsKKyAgICAgICAgc3BtdmJlbmNoOjpzcG1tX2Nzcl90aWxlZF9mMzIoQSwgQiwgaywgdGlsZSwgYmFsYW5jZWQsIC8qaXRlcnM9Ki8xLAorICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAvKndhcm11cD0qLzAsIFkpOworICAgICAgfSBlbHNlIHsKKyAgICAgICAgc3BtdmJlbmNoOjpzcG1tX2Nzcl90aWxlZF9mNjQoQSwgQiwgaywgdGlsZSwgYmFsYW5jZWQsIC8qaXRlcnM9Ki8xLAorICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAvKndhcm11cD0qLzAsIFkpOworICAgICAgfQorICAgICAgQVNTRVJUX0VRKFkuc2l6ZSgpLCBZX3JlZi5zaXplKCkpOworICAgICAgc3RkOjp2ZWN0b3I8QWNjPiBZZ3B1KFkuc2l6ZSgpKTsKKyAgICAgIGZvciAoc3RkOjpzaXplX3QgaSA9IDA7IGkgPCBZLnNpemUoKTsgKytpKQorICAgICAgICBZZ3B1W2ldID0gc3RhdGljX2Nhc3Q8QWNjPihZW2ldKTsKKyAgICAgIGNoZWNrX3NwbW08VCwgQWNjPihuYW1lLAorICAgICAgICAgICAgICAgICAgICAgICAgICJzcG1tX3RpbGVkX3RpbGUiICsgc3RkOjp0b19zdHJpbmcodGlsZSkgKworICAgICAgICAgICAgICAgICAgICAgICAgICAgICAoYmFsYW5jZWQgPyAiX2J1Y2tldGVkIiA6ICJfdW5pZm9ybSIpLAorICAgICAgICAgICAgICAgICAgICAgICAgIEEsIEIsIGssIFlfcmVmLCBZZ3B1KTsKKyAgICB9CisgIH0KK30KKworfSAgLy8gbmFtZXNwYWNlCisKK2NsYXNzIEN1ZGFTcG12Q29ycmVjdG5lc3MgOiBwdWJsaWMgOjp0ZXN0aW5nOjpUZXN0V2l0aFBhcmFtPHN0ZDo6c3RyaW5nPiB7fTsKKworVEVTVF9QKEN1ZGFTcG12Q29ycmVjdG5lc3MsIEZwMzIpIHsKKyAgcnVuX3NwbXZfbWF0cml4PGZsb2F0LCBmbG9hdD4oR2V0UGFyYW0oKSk7Cit9CisKK1RFU1RfUChDdWRhU3BtdkNvcnJlY3RuZXNzLCBGcDY0KSB7CisgIHJ1bl9zcG12X21hdHJpeDxkb3VibGUsIGRvdWJsZT4oR2V0UGFyYW0oKSk7Cit9CisKK0lOU1RBTlRJQVRFX1RFU1RfU1VJVEVfUChTeW50aGV0aWNNYXRyaWNlcywgQ3VkYVNwbXZDb3JyZWN0bmVzcywKKyAgICAgICAgICAgICAgICAgICAgICAgICA6OnRlc3Rpbmc6OlZhbHVlc0luKHN5bnRoZXRpY19tYXRyaWNlcygpKSk7CisKK2NsYXNzIEN1ZGFTcG1tQ29ycmVjdG5lc3MgOiBwdWJsaWMgOjp0ZXN0aW5nOjpUZXN0V2l0aFBhcmFtPHN0ZDo6c3RyaW5nPiB7fTsKKworVEVTVF9QKEN1ZGFTcG1tQ29ycmVjdG5lc3MsIEZwMzIpIHsKKyAgcnVuX3NwbW1fbWF0cml4PGZsb2F0LCBmbG9hdD4oR2V0UGFyYW0oKSk7Cit9CisKK1RFU1RfUChDdWRhU3BtbUNvcnJlY3RuZXNzLCBGcDY0KSB7CisgIHJ1bl9zcG1tX21hdHJpeDxkb3VibGUsIGRvdWJsZT4oR2V0UGFyYW0oKSk7Cit9CisKK0lOU1RBTlRJQVRFX1RFU1RfU1VJVEVfUChTeW50aGV0aWNNYXRyaWNlcywgQ3VkYVNwbW1Db3JyZWN0bmVzcywKKyAgICAgICAgICAgICAgICAgICAgICAgICA6OnRlc3Rpbmc6OlZhbHVlc0luKHN5bnRoZXRpY19tYXRyaWNlcygpKSk7CisKKy8vIFRpbGUtYm91bmRhcnkgY292ZXJhZ2U6IGsgc21hbGxlciB0aGFuLCBlcXVhbCB0bywgYW5kIGxhcmdlciB0aGFuIHRoZSB0aWxlcy4KK1RFU1QoQ3VkYVNwbW1FZGdlSywgVGlueUsxKSB7CisgIHJ1bl9zcG1tX21hdHJpeDxkb3VibGUsIGRvdWJsZT4oInRpbnlfM3gzIiwgMSk7Cit9CitURVNUKEN1ZGFTcG1tRWRnZUssIFRpbnlLMTUpIHsKKyAgcnVuX3NwbW1fbWF0cml4PGRvdWJsZSwgZG91YmxlPigidGlueV8zeDMiLCAxNSk7Cit9CitURVNUKEN1ZGFTcG1tRWRnZUssIFRpbnlLNjQpIHsKKyAgcnVuX3NwbW1fbWF0cml4PGRvdWJsZSwgZG91YmxlPigidGlueV8zeDMiLCA2NCk7Cit9CisKKyNlbmRpZiAgLy8gU1BNVkJFTkNIX0VOQUJMRV9DVURBCg==
"""

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/AbhiramCommits/SpMVBench.git", REPO], check=True)
os.chdir(REPO)
# Pin the base commit so the patch always applies cleanly.
subprocess.run(["git", "checkout", "--quiet", "--force", BASE], check=True)
subprocess.run(["git", "clean", "-fdq"], check=True)
pathlib.Path("/tmp/spmvbench_cuda.patch").write_bytes(base64.b64decode(PATCH_B64))
rc = subprocess.run(["git", "apply", "--whitespace=nowarn",
                     "/tmp/spmvbench_cuda.patch"]).returncode
print("git apply exit code:", rc)
if rc != 0:
    raise SystemExit("PATCH FAILED TO APPLY")
print("SOURCE READY at", REPO)


In [ ]:
import os, subprocess, sys

os.chdir("/content/SpMVBench")


def run(cmd, check=True):
    print("\n" + "=" * 78)
    print("$", cmd)
    print("=" * 78, flush=True)
    p = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout, flush=True)
    if check and p.returncode != 0:
        raise SystemExit("COMMAND FAILED (exit %d): %s" % (p.returncode, cmd))
    return p.returncode


# ---- 1. real device / toolchain information -------------------------------
run("uname -a")
run("nvidia-smi", check=False)
run("nvcc --version")
run("nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap "
    "--format=csv", check=False)
run("nproc")

# ---- 2. compute capability -> CMAKE_CUDA_ARCHITECTURES --------------------
cc = None
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        text=True, stderr=subprocess.STDOUT).strip().splitlines()[0].strip()
    if out and "N/A" not in out and "." in out:
        cc = out.replace(".", "")
except Exception as exc:
    print("nvidia-smi compute_cap query failed:", exc)

if cc is None:
    try:
        import torch
        major, minor = torch.cuda.get_device_capability()
        cc = f"{major}{minor}"
    except Exception as exc:
        print("torch fallback failed:", exc)

if cc is None:
    raise SystemExit(
        "Could not determine the GPU compute capability. "
        "Please set CMAKE_CUDA_ARCHITECTURES manually.")

print("\n>>> Using CMAKE_CUDA_ARCHITECTURES =", cc)

# ---- 3. data + build + tests ---------------------------------------------
run("python3 -c \"import numpy, scipy; print('numpy', numpy.__version__, "
    "'scipy', scipy.__version__)\"", check=False)
run("python3 tools/gen_matrices.py")
run("python3 tools/reference.py")

run("cmake -S . -B build -DSPMVBENCH_ENABLE_CUDA=ON "
    "-DCMAKE_CUDA_ARCHITECTURES=" + cc + " -DCMAKE_BUILD_TYPE=Release")
run("cmake --build build -j")

rc = run("ctest --test-dir build --output-on-failure", check=False)
print("\n>>> ctest exit code:", rc)
